# Traffic Source Quality (Source / Medium)

This notebook compares acquisition channels by traffic quality, not just volume.

### Quick start (beginner-friendly)
1. Run the **Setup (run once)** cell.
2. In **Parameters**, set `DAYS_BACK` and `TOP_N`.
3. Run the remaining cells from top to bottom.

### Links
- GitHub repo: [github.com/aidanm-lla/lla-data](https://github.com/aidanm-lla/lla-data)
- Open this notebook in Colab: [Traffic Source Quality](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/05_traffic_sources.ipynb)

### Other notebooks
- [Search Contribution Overview](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/01_search_contribution_overview.ipynb)
- [Top Pages Search Performance](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/02_top_pages_search_performance.ipynb)
- [Query Drivers by Page](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/03_query_drivers_by_page.ipynb)
- [Opportunity Watchlist](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/04_opportunity_watchlist.ipynb)
- [Top Pages (Last 7 Days)](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/top_pages_last_7_days.ipynb)
- [Time Patterns for Crisis-Related Pages](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/06_time_patterns.ipynb)
- [Crisis Support Funnel](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/crisis_funnel.ipynb)
- [Analysis Template](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/templates/analysis_template.ipynb)

**Metrics:** sessions, engagement rate, average pages per session
**Data source:** GA4 BigQuery export (`events_*`)


In [ ]:
#@title Setup (run once)
import sys
import os

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidoanto/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q 'plotly>=6.1.1' 'kaleido>=1.2.0'
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import pandas as pd
import plotly.express as px

import lifeline_theme
from lla_data import config
from lla_data.bq import build_date_params, default_query_window, get_client, load_sql_template, run_query

lifeline_theme.inject_fonts()

client = get_client()

In [ ]:
#@title Parameters
DAYS_BACK = 35 #@param {type:"integer"}
TOP_N = 12 #@param {type:"integer"}
MINIMUM_SESSIONS = 20 #@param {type:"integer"}

window = default_query_window(days_back=DAYS_BACK)

In [ ]:
from google.cloud import bigquery

query = load_sql_template(
    "sql/notebooks/ga4_traffic_sources_summary.sql",
    project_id=config.PROJECT_ID,
    ga4_dataset=config.GA4_DATASET,
)

analysis_end = window.end_date
analysis_start = window.start_date
analysis_window_label = f"{analysis_start:%Y-%m-%d} to {analysis_end:%Y-%m-%d}"

params = build_date_params(window) + [
    bigquery.ScalarQueryParameter("minimum_sessions", "INT64", MINIMUM_SESSIONS),
]

df = run_query(client, query, params=params)
df["source_medium"] = df["source"] + " / " + df["medium"]
df.head()

In [ ]:
top_df = df.nlargest(TOP_N, "sessions").sort_values("sessions", ascending=True)

fig = px.bar(
    top_df,
    x="sessions",
    y="source_medium",
    orientation="h",
    template="lifeline",
    title=f"Top {TOP_N} Source/Medium by Sessions ({analysis_window_label})",
    labels={"source_medium": "Source / Medium", "sessions": "Sessions"},
)
fig.update_layout(height=560, margin=dict(l=170, r=40, t=80, b=60))
fig.update_yaxes(automargin=True)
lifeline_theme.add_lifeline_logo(fig)
fig.show()

In [ ]:
# Limit to top 20 sources by sessions
quality_df = df[df["sessions"] >= 50].copy()
quality_df = quality_df.nlargest(20, "sessions")

fig = px.scatter(
    quality_df,
    x="engagement_rate",
    y="avg_pages_per_session",
    size="sessions",
    size_max=35,
    color="source",
    hover_name="source_medium",
    template="lifeline",
    title=f"Top 20 Source/Medium – Acquisition Quality Map ({analysis_window_label})",
    labels={
        "engagement_rate": "Engagement Rate",
        "avg_pages_per_session": "Average Pages per Session",
    },
)

# Optional: tweak sizeref to further adjust sizes if still too large or small
max_sessions = quality_df["sessions"].max()
sizeref = 2.*max_sessions/(100**2)
fig.update_traces(marker=dict(sizeref=sizeref, sizemode="area"))

fig.update_xaxes(tickformat=".0%")
lifeline_theme.add_lifeline_logo(fig)
fig.show()